# Legacy Figure-Generation Pipelines (deprecated, non-paper)

Split out of `audit_pipeline/notebooks/figures_consolidated.ipynb` on
2026-09-10, when that notebook was narrowed to contain only the 14 figures
actually embedded in the paper. This notebook reproduces the two
figure-generation code paths that are **not** used by the paper --
`audit_pipeline/stage5_figures.py` (archived; superseded by
`generate_figures_final.py`'s logic, which is what `figures_consolidated.ipynb`
now contains) and the deprecated `auditing/03_audit_visualizations.ipynb`
pipeline (archived; structurally incompatible with `audit_pipeline`, no
`coding_level` stratification) -- plus the one-time Step 3 cross-stage
consistency check that validated both of them against the live
`audit_pipeline` stage1-4 outputs.

**Nothing here produces a paper figure.** Every output below writes under
`deprecated/legacy_outputs/`, never under `outputs/`, so that `outputs/`
holds only paper-relevant content. `outputs/stage1`-`outputs/stage4` are
still read (read-only) for Step 3's comparisons, since those are the live,
current pipeline outputs being validated against.

See `deprecated/README.md` for why `stage5_figures.py` and the `auditing/`
pipeline are archived, and `figures_consolidated.ipynb`'s own history
(the "Style Pass Summary" and "Addendum" cells) for how the paper-figure
half of this split notebook reached its current state.

## Setup -- imports and repository paths

Same repo-root discovery as `figures_consolidated.ipynb`. Unlike that
notebook, this one still needs `audit_pipeline.config`/`audit_pipeline.helpers`
(`stage5_figures.py`'s reproduction is `PipelineVariant`-driven) and reads
from the real `outputs/stage1`-`outputs/stage4` (read-only) for Step 3's
cross-stage comparisons -- only this notebook's own *writes* are redirected
away from `outputs/`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "audit_pipeline").is_dir():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Could not locate repo root (no audit_pipeline/ found above cwd).")
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUTPUTS_DIR = REPO_ROOT / "outputs"                          # read-only: stage1-4 inputs for Step 3
LEGACY_OUTPUTS_DIR = REPO_ROOT / "deprecated" / "legacy_outputs"  # everything this notebook writes

import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, Normalize
from pandas.errors import EmptyDataError

from audit_pipeline.config import (
    DI_THRESHOLD,
    N_MIN,
    VARIANT_FULL,
    VARIANT_TIER12,
    PipelineVariant,
)
from audit_pipeline.helpers import map_reporting_group, norm_target

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"OUTPUTS_DIR (read-only) = {OUTPUTS_DIR}")
print(f"LEGACY_OUTPUTS_DIR (writes) = {LEGACY_OUTPUTS_DIR}")

# stage5_figures.py's reproduction below writes through variant.out_s5;
# redirect that one field to the legacy output tree. stage5_figures.py
# itself is archived (deprecated/stage5_figures.py) and outputs/stage5/
# no longer exists as a live pipeline output.
variant = VARIANT_FULL._replace(out_s5=LEGACY_OUTPUTS_DIR / "stage5")

## `audit_pipeline/stage5_figures.py` -- shared utilities

Copied verbatim from `stage5_figures.py` (module-level style block, palette
constants, and helper functions), with a `_s5` suffix added to every helper
name (`_read` -> `_read_s5`, `_save` -> `_save_s5`, etc.) purely to avoid
colliding with `generate_figures_final.py`'s own same-named helpers, which
are semantically different (different DPI, different file format, different
`tight_layout` handling) and are kept separately below rather than merged --
merging them would be a behavior change, not a consolidation.

One line was added to `_save_s5` (not present in the original
`stage5_figures.py`): a `plt.show()` call right before `plt.close(fig)`, so
each figure renders inline in this notebook's cell output as it's produced.
Marked inline with an `# added:` comment. Nothing about the figure itself
(data, layout, threshold, colors) is touched.

**2026-07-03 style pass:** the six palette constants (`_BLUE`, `_GREEN`,
`_ORANGE`, `_RED`, `_PURPLE`, `_TEAL`) and the L2-L4 entries of `_CL_COLOR`
were changed to the exact hex values `generate_figures_final.py`'s
`STYLE["colors"]` already uses, so "correct"/"case A" is the same green,
"failure"/"case B"/"fail" is the same vermillion, and "presence"/"pass" is
the same blue in every figure in this notebook, not just within one figure
family. Only the literal hex values changed at their single definition
point -- every figure function below still references these same constant
names, so no data, threshold, sort, or filter logic moved. `L1` keeps its
own accent color since `generate_figures_final.py` figures never facet on
L1. See the constants cell below for the full old-vs-new mapping.


In [ ]:
# ── Global style ──────────────────────────────────────────────────────────────
mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.labelsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.8,
        "axes.grid": True,
        "axes.grid.axis": "x",
        "grid.color": "#e0e0e0",
        "grid.linewidth": 0.6,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "legend.framealpha": 0.7,
        "legend.edgecolor": "#cccccc",
        "figure.facecolor": "white",
        "axes.facecolor": "#fafafa",
        "savefig.facecolor": "white",
    }
)

# Shared palettes
# 2026-07-03 style pass: aligned to generate_figures_final.py's STYLE["colors"]
# (Okabe-Ito colorblind-safe palette) so the same semantic category -- pass/
# presence, correct/case A, fail/failure/case B/negative-delta -- renders as
# the same hex wherever it appears across BOTH figure families, not just
# within one. Only the literal hex values changed; every call site below
# keeps referencing these same constant names, so no plotting logic moved.
_BLUE = "#0072B2"      # was #4C78A8 -- now matches STYLE["colors"]["presence"/"pass"]
_GREEN = "#009E73"     # was #54A24B -- now matches STYLE["colors"]["correct"/"case_a"]
_ORANGE = "#D55E00"    # was #F28E2B -- now matches STYLE["colors"]["failure"/"case_b"/"fail"]
_RED = "#D55E00"       # was #E15759 -- now matches STYLE["colors"]["fail"/"delta_neg"] (never
                       # co-occurs with _ORANGE in the same figure, so sharing this hex is safe)
_PURPLE = "#CC79A7"    # was #B279A2 -- now matches STYLE["colors"]["L4"]
_TEAL = "#56B4E9"      # was #72B7B2 -- now matches STYLE["colors"]["L3"/"type_cov"]

# Coding-level accent colors used for facet titles / highlights.
# L2-L4 aligned to generate_figures_final.py's STYLE["colors"]; L1 has no gff
# counterpart (gff figures never facet on L1) so it keeps its own accent.
_CL_COLOR: dict[str, str] = {
    "L1": "#E15759",  # s5-only level; no canonical gff color to align to
    "L2": "#E69F00",  # was #F28E2B -- now matches STYLE["colors"]["L2"]
    "L3": "#56B4E9",  # was #4C78A8 -- now matches STYLE["colors"]["L3"]
    "L4": "#CC79A7",  # was #B279A2 -- now matches STYLE["colors"]["L4"]
}


def _cl_color_s5(cl: str) -> str:
    """Return the accent color assigned to a coding level label."""
    return _CL_COLOR.get(str(cl), "#777777")

def _read_s5(path):
    """Read a TSV artifact, returning an empty frame when absent or empty."""
    try:
        return pd.read_csv(path, sep="\t", low_memory=False)
    except (FileNotFoundError, EmptyDataError):
        return pd.DataFrame()

def _save_s5(fig, path):
    """Save a figure with shared layout and export settings."""
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()  # added: display the figure inline in the notebook output before closing it
    plt.close(fig)

def _filter_self_ref_s5(df: pd.DataFrame) -> pd.DataFrame:
    """Drop self-referential dogwhistle rows when the flag is available."""
    if "is_self_referential" not in df.columns:
        return df.copy()
    return df[~df["is_self_referential"].fillna(False)].copy()

def _top_n_s5(df: pd.DataFrame, by: str, n: int, ascending: bool = False) -> pd.DataFrame:
    """Return the top ``n`` rows by a metric column if that column exists.

    ``ascending=False`` (default) selects the *highest* n values -- correct
    for metrics where higher = more extreme (token frequency, label gaps,
    etc.). Pass ``ascending=True`` for metrics where *lower* = worse (e.g.
    DI ratios), so "top n" means "n most disparate", not "n with the highest
    raw value".
    """
    if df.empty or by not in df.columns:
        return df
    return df.sort_values(by, ascending=ascending).head(n).copy()

def _style_ax_s5(ax, grid_axis: str = "x") -> None:
    """Remove top/right spines, set tick params."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(length=3, width=0.7)
    ax.grid(True, axis=grid_axis, color="#e0e0e0", linewidth=0.6, zorder=0)

def _facet_title_s5(ax, label: str) -> None:
    """Colour-coded panel title for coding-level facets."""
    color = _cl_color_s5(label)
    ax.set_title(label, color=color, fontweight="bold", fontsize=11, pad=6, loc="left")


### Level-stratified figures (`stage5_figures.level_stratified_figures`)

**Function:** `level_stratified_figures(level_dir, s1_dir, s2_dir, s3_dir)`

**Source TSVs read** (all via `variant.out_sN`, i.e. Stage 1-3 of
`audit_pipeline`, primary/"full" variant by default):
- `stage1/s1_coverage_by_level_target.tsv` -- **Stage 1**
- `stage2/s2_annotation_by_level_target.tsv` -- **Stage 2**
- `stage3/s3_coverage_disparity.tsv`, `stage3/s3_annotation_disparity.tsv`,
  `stage3/s3_cross_level_consistency.tsv` -- **Stage 3**

**Output PNGs** (under `variant.out_s5/level_stratified/`):
`s5_lev_coverage_presence_vs_type.png`, `s5_lev_coverage_token_frequency.png`,
`s5_lev_annotation_rates.png`, `s5_lev_annotation_case_ab.png`,
`s5_lev_disparity_di_histograms.png`, `s5_lev_disparity_worst_di.png`,
`s5_lev_annotation_label_gap.png`, `s5_lev_cross_level_deltas.png`.

Note the worst-DI figure here (`s5_lev_disparity_worst_di.png`) reads its
pairwise disparity numbers from **Stage 3** (`target_a`/`target_b` are raw,
un-collapsed taxonomy targets) -- contrast with the group-collapsed section
below, whose worst-DI figure reads **Stage 4a** (`by_group`) instead, and
with `generate_figures_final.py`'s equivalent figures (`fig4`/`fig5`), which
read **Stage 4b** (`by_level_group`). All three exist in this repo and none
of them agree on which source file is "the" pairwise DI source -- this is
exactly the kind of divergence Step 3 below checks numerically.

Previously contained the `_top_n_s5` sort-direction bug at the
`_top_n_s5(sub, "worst_di_ratio", 8)` call -- fixed: this call now
passes `ascending=True` so the n MOST disparate pairs are selected.


In [ ]:
def level_stratified_figures(
    level_dir: Path,
    s1_dir: Path,
    s2_dir: Path,
    s3_dir: Path,
) -> None:
    """Render the coding-level faceted figure set from Stages 1-3 outputs.

    Parameters
    ----------
    level_dir : Path
        Output directory for level-stratified figures.
    s1_dir, s2_dir, s3_dir : Path
        Input directories for the corresponding stage artifacts.
    """
    s1 = _filter_self_ref_s5(_read_s5(s1_dir / "s1_coverage_by_level_target.tsv"))
    s2 = _filter_self_ref_s5(_read_s5(s2_dir / "s2_annotation_by_level_target.tsv"))
    s3c = _read_s5(s3_dir / "s3_coverage_disparity.tsv")
    s3a = _read_s5(s3_dir / "s3_annotation_disparity.tsv")
    s3x = _read_s5(s3_dir / "s3_cross_level_consistency.tsv")

    def _coding_levels(df):
        """Return coding levels in paper order, dropping absent facets."""
        return [
            c
            for c in ["L1", "L2", "L3", "L4"]
            if c in set(df["coding_level"].dropna().astype(str))
        ]

    if not s1.empty:
        cls = _coding_levels(s1)

        # -- Coverage: presence rate + type coverage, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.2), sharey=False)  # was (14, 10); matches gff textwidth_in
        fig.suptitle(
            "Coverage — Presence Rate and Type Coverage by Target Group",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            d = _top_n_s5(
                s1[s1["coding_level"].astype(str) == cl], "token_frequency", 10
            ).copy()
            d["label"] = d["target"].astype(str)
            ax = axes.flatten()[i]
            y = range(len(d))
            ax.barh(
                y,
                d["presence_rate"],
                color=_cl_color_s5(cl),
                alpha=0.85,
                label="Presence rate",
                edgecolor="white",
                linewidth=0.5,
            )
            ax.barh(
                y,
                d["type_coverage"],
                color=_GREEN,
                alpha=0.55,
                label="Type coverage",
                edgecolor="white",
                linewidth=0.5,
            )
            ax.set_yticks(list(y))
            ax.set_yticklabels(d["label"], fontsize=8)
            ax.invert_yaxis()
            ax.set_xlim(0, 1.05)
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="x")
            ax.legend(loc="lower right", fontsize=7)
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_coverage_presence_vs_type.png")

        # -- Coverage: token frequency top-10, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.2), sharey=False)  # was (14, 10); matches gff textwidth_in
        fig.suptitle(
            "Coverage — Top-10 Token Frequency by Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            d = _top_n_s5(s1[s1["coding_level"].astype(str) == cl], "token_frequency", 10)
            ax = axes.flatten()[i]
            ax.barh(
                d["target"].astype(str),
                d["token_frequency"],
                color=_cl_color_s5(cl),
                edgecolor="white",
                linewidth=0.5,
            )
            ax.invert_yaxis()
            ax.set_xlabel("Token frequency", fontsize=8)
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="x")
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_coverage_token_frequency.png")

    if not s2.empty:
        cls = _coding_levels(s2)

        # -- Annotation rates, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.5), sharey=False)  # was (15, 10); matches gff textwidth_in
        fig.suptitle(
            "Annotation Quality — Correct vs. Failure Rate by Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            d = _top_n_s5(
                s2[s2["coding_level"].astype(str) == cl], "total_matches", 8
            ).copy()
            d["label"] = (
                d["target"].astype(str) + " (" + d["coding_level"].astype(str) + ")"
            )
            ax = axes.flatten()[i]
            x = range(len(d))
            bar_w = 0.38
            ax.bar(
                x,
                d["correct_labeling_rate"],
                width=bar_w,
                label="Correct",
                color=_GREEN,
                edgecolor="white",
                linewidth=0.5,
            )
            ax.bar(
                [t + bar_w for t in x],
                d["failure_rate"],
                width=bar_w,
                label="Failure",
                color=_ORANGE,
                edgecolor="white",
                linewidth=0.5,
            )
            ax.set_xticks([t + bar_w / 2 for t in x])
            ax.set_xticklabels(d["label"], rotation=40, ha="right", fontsize=8)
            ax.set_ylim(0, 1.05)
            ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
            ax.set_ylabel("Rate", fontsize=8)
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="y")
            ax.legend(loc="upper right")
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_annotation_rates.png")

        # -- Case A/B stacked, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.5), sharey=False)  # was (15, 10); matches gff textwidth_in
        fig.suptitle(
            "Annotation Quality — Case A vs. B Match Counts by Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            d = _top_n_s5(
                s2[s2["coding_level"].astype(str) == cl], "total_matches", 8
            ).copy()
            d["label"] = (
                d["target"].astype(str) + " (" + d["coding_level"].astype(str) + ")"
            )
            ax = axes.flatten()[i]
            ax.barh(
                d["label"],
                d["case_a_present_hateful"],
                color=_GREEN,
                label="Case A (hateful)",
                edgecolor="white",
                linewidth=0.5,
            )
            ax.barh(
                d["label"],
                d["case_b_present_nonhateful"],
                left=d["case_a_present_hateful"],
                color=_ORANGE,
                label="Case B (non-hateful)",
                edgecolor="white",
                linewidth=0.5,
            )
            ax.invert_yaxis()
            ax.set_xlabel("Match count", fontsize=8)
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="x")
            ax.legend(loc="lower right")
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_annotation_case_ab.png")

    if not s3c.empty:
        cls = _coding_levels(s3c)

        # -- DI ratio histograms, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.2), sharey=False)  # was (14, 10); matches gff textwidth_in
        fig.suptitle(
            "Disparity — Presence & Type Coverage DI Ratio Distributions by Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            d = s3c[s3c["coding_level"].astype(str) == cl]
            ax = axes.flatten()[i]
            ax.hist(
                d["presence_rate_di_ratio"].dropna(),
                bins=15,
                alpha=0.75,
                color=_BLUE,
                edgecolor="white",
                linewidth=0.5,
                label="Presence DI",
            )
            ax.hist(
                d["type_coverage_di_ratio"].dropna(),
                bins=15,
                alpha=0.55,
                color=_TEAL,
                edgecolor="white",
                linewidth=0.5,
                label="Type DI",
            )
            ax.axvline(
                DI_THRESHOLD,
                color=_RED,
                linestyle="--",
                linewidth=1.2,
                label=f"4/5 threshold ({DI_THRESHOLD})",
            )
            ax.set_xlabel("DI ratio", fontsize=8)
            ax.set_ylabel("Count", fontsize=8)
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="y")
            ax.legend()
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_disparity_di_histograms.png")

        # -- Worst DI by pair, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.2), sharey=False)  # was (14, 10); matches gff textwidth_in
        fig.suptitle(
            "Disparity — Worst Coverage DI Ratio by Target Pair and Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            sub = s3c[s3c["coding_level"].astype(str) == cl].copy()
            sub["pair"] = (
                sub["target_a"].astype(str) + " vs " + sub["target_b"].astype(str)
            )
            # FIXED (see stage5_figures.py c92ea92-series fix): _top_n_s5() now takes
            # an explicit ascending= arg; "worst_di_ratio" (lower = more disparate =
            # worse) passes ascending=True so the n MOST disparate pairs are selected.
            d = _top_n_s5(sub, "worst_di_ratio", 8, ascending=True).sort_values(
                "worst_di_ratio", ascending=True
            )
            colors = [_RED if v < DI_THRESHOLD else _BLUE for v in d["worst_di_ratio"]]
            ax = axes.flatten()[i]
            ax.barh(
                d["pair"],
                d["worst_di_ratio"],
                color=colors,
                edgecolor="white",
                linewidth=0.5,
            )
            ax.axvline(
                DI_THRESHOLD,
                color=_RED,
                linestyle="--",
                linewidth=1.2,
                label=f"4/5 rule ({DI_THRESHOLD})",
            )
            ax.set_xlim(0, 1.05)
            ax.set_xlabel("Worst DI ratio", fontsize=8)
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="x")
            ax.legend(fontsize=7)
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_disparity_worst_di.png")

    if not s3a.empty:
        cls = _coding_levels(s3a)

        # -- Annotation label gap, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.2), sharey=False)  # was (14, 10); matches gff textwidth_in
        fig.suptitle(
            "Disparity — Annotation Labeling Rate Gap by Target Pair and Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            sub = s3a[s3a["coding_level"].astype(str) == cl].copy()
            sub["pair"] = (
                sub["target_a"].astype(str) + " vs " + sub["target_b"].astype(str)
            )
            d = _top_n_s5(sub, "labeling_rate_gap_abs", 8).sort_values(
                "labeling_rate_gap_abs", ascending=True
            )
            ax = axes.flatten()[i]
            ax.barh(
                d["pair"],
                d["labeling_rate_gap_abs"],
                color=_RED,
                edgecolor="white",
                linewidth=0.5,
            )
            ax.set_xlabel("|Correct rate gap|", fontsize=8)
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="x")
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_annotation_label_gap.png")

    if not s3x.empty:
        d = s3x.copy()
        d["transition"] = (
            d["target"].astype(str)
            + " ("
            + d["taxonomy_level"].astype(str)
            + "): "
            + d["coding_level_from"].astype(str)
            + " \u2192 "
            + d["coding_level_to"].astype(str)
        )
        d = _top_n_s5(
            d.assign(abs_presence=d["presence_rate_delta"].abs()), "abs_presence", 20
        )
        fig, ax = plt.subplots(figsize=(6.97, 6.5))  # was (13, 8); matches gff textwidth_in
        fig.suptitle(
            "Cross-Level Consistency — Presence and Labeling Rate \u0394 by Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        y = range(len(d))
        ax.barh(
            y,
            d["presence_rate_delta"],
            color=_BLUE,
            alpha=0.85,
            label="Presence \u0394",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.barh(
            y,
            d["correct_labeling_rate_delta"],
            color=_GREEN,
            alpha=0.65,
            label="Labeling \u0394",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.set_yticks(list(y))
        ax.set_yticklabels(d["transition"], fontsize=8)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_xlabel("Delta", fontsize=9)
        _style_ax_s5(ax, grid_axis="x")
        ax.legend()
        _save_s5(fig, level_dir / "s5_lev_cross_level_deltas.png")


In [ ]:
level_stratified_figures(
    variant.out_s5 / "level_stratified",
    variant.out_s1,
    variant.out_s2,
    variant.out_s3,
)


### Group-collapsed figures (`stage5_figures.group_collapsed_figures`)

**Function:** `group_collapsed_figures(group_dir, s4_dir)`

**Source TSVs read** (mixed within a single function -- see note below):
- `stage4/by_level_group/s4b_coverage_by_level_group.tsv` -- **Stage 4b**
  (used for the coverage heatmap only)
- `stage4/by_group/s4a_annotation_by_group.tsv` -- **Stage 4a**
  (annotation rates + Case A/B counts)
- `stage4/by_group/s4a_pairwise_disparity_by_group.tsv` -- **Stage 4a**
  (DI histograms + worst-DI/label-gap panel)

**Output PNGs** (under `variant.out_s5/group_collapsed/`):
`s5_grp_coverage_scatter.png`, `s5_grp_annotation_rates.png`,
`s5_grp_case_ab_counts.png`, `s5_grp_di_histograms.png`,
`s5_grp_worst_di_and_label_gap.png`.

**This is the exact ambiguity called out in the task brief**: the coverage
heatmap in this same function reads Stage 4**b** (`by_level_group` -- the
file a reviewer checking taxonomy-preserving numbers would likely reach for
first), while the pairwise-disparity figures three lines later, in the same
function, read Stage 4**a** (`by_group`) instead -- and, per the section
above, the level-stratified pairwise figure reads **Stage 3**. Three
different pairwise-DI sources across two sections of this one file.

Previously contained the `_top_n_s5` sort-direction bug at
`_top_n_s5(pair_plot, "worst_di_ratio", 20)` -- fixed: this call now
passes `ascending=True` so the n MOST disparate pairs are selected.


In [ ]:
def group_collapsed_figures(group_dir: Path, s4_dir: Path) -> None:
    """Render reporting-group figures from Stage 4 collapsed outputs.

    Parameters
    ----------
    group_dir : Path
        Output directory for group-collapsed figures.
    s4_dir : Path
        Input directory containing Stage 4 artifacts.
    """
    # Use by-level-group coverage so duplicated taxonomy mappings can be deduplicated
    # explicitly by report group + coding level for a stable heatmap matrix.
    cov = _read_s5(s4_dir / "by_level_group/s4b_coverage_by_level_group.tsv")
    ann = _filter_self_ref_s5(_read_s5(s4_dir / "by_group/s4a_annotation_by_group.tsv"))
    pair = _read_s5(s4_dir / "by_group/s4a_pairwise_disparity_by_group.tsv")

    if not cov.empty:
        d = cov.copy()
        d["group_label"] = (
            d["report_level"].astype(str).str.strip()
            + ": "
            + d["report_target"].astype(str).str.strip()
        )
        d["coding_level"] = d["coding_level"].astype(str).str.strip()

        # Keep only primary coding levels used in paper-facing visuals.
        d = d[d["coding_level"].isin(["L2", "L3", "L4"])].copy()
        # Some groups may appear in multiple taxonomy rows; keep first group-level row.
        d = d.drop_duplicates(subset=["group_label", "coding_level"], keep="first")

        group_order = [
            "disability: unspecific",
            "gender: men",
            "gender: women",
            "lgbtq: LGB",
            "lgbtq: Trans/NB",
            "origin: specific country",
            "origin: undocumented",
            "origin: immigrant",
            "origin: migrant worker",
            "politics: communist",
            "politics: democrat",
            "politics: libertarian",
            "politics: leftist",
            "politics: liberal",
            "politics: conservative",
            "politics: republican",
            "race: asian",
            "race: black",
            "race: latinx",
            "race: middle eastern",
            "race: white",
            "religion: jewish",
            "religion: muslim",
        ]
        existing_groups = set(d["group_label"].astype(str))
        group_labels = [g for g in group_order if g in existing_groups]
        remaining = sorted(existing_groups - set(group_labels))
        group_labels.extend(remaining)
        coding_levels = ["L2", "L3", "L4"]

        presence_mat = d.pivot(
            index="group_label", columns="coding_level", values="presence_rate"
        ).reindex(index=group_labels, columns=coding_levels)
        type_mat = d.pivot(
            index="group_label", columns="coding_level", values="type_coverage"
        ).reindex(index=group_labels, columns=coding_levels)

        # Distinguish structural missing (NaN) from true 0% presence cells.
        cmap = LinearSegmentedColormap.from_list(
            "presence_rate",
            [
                (0.0, "#CBD4D0"),
                (0.35, "#8FC6A2"),
                (0.7, "#3B956F"),
                (1.0, "#14523A"),
            ],
        )
        cmap.set_bad(color="#E8E8E8")

        fig_h = max(7, len(group_labels) * 0.42 + 2.8)
        fig, ax = plt.subplots(figsize=(6.97, fig_h))  # was 8.4; matches gff textwidth_in
        fig.suptitle(
            "Coverage Heatmap — Presence Rate by Reporting Group × Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.99,
        )

        data = presence_mat.values.astype(float)
        masked = np.ma.array(data, mask=np.isnan(data))
        im = ax.imshow(
            masked,
            aspect="auto",
            cmap=cmap,
            vmin=0.0,
            vmax=1.0,
            interpolation="none",
        )

        flagged = []
        for r, grp in enumerate(group_labels):
            for c, cl in enumerate(coding_levels):
                pr = presence_mat.loc[grp, cl]
                tc = type_mat.loc[grp, cl]

                if pd.isna(pr):
                    ax.text(
                        c,
                        r,
                        "-",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color="#8a8a8a",
                    )
                    continue

                has_type_gap = pd.notna(tc) and 0.0 < tc < 0.999
                txt_color = "white" if pr > 0.5 else "#1a1a1a"
                label = f"{pr:.0%}"
                if has_type_gap:
                    label = label + "$^{*}$"
                ax.text(
                    c,
                    r,
                    label,
                    ha="center",
                    va="center",
                    fontsize=8.5,
                    fontweight="bold",
                    color=txt_color,
                )

                if has_type_gap:
                    flagged.append(
                        f"{grp} / {cl}: type_coverage={tc:.3f}, presence_rate={pr:.3f}"
                    )
                    ax.add_patch(
                        mpatches.FancyBboxPatch(
                            (c - 0.47, r - 0.47),
                            0.94,
                            0.94,
                            boxstyle="square,pad=0",
                            linewidth=1.9,
                            edgecolor="#C0392B",
                            facecolor="none",
                            zorder=3,
                        )
                    )

        ax.set_xticks(range(len(coding_levels)))
        ax.set_xticklabels(
            [
                "L2\n(Stereotype-based)",
                "L3\n(Concept / policy)",
                "L4\n(Persona signals)",
            ],
            fontsize=8.5,
        )
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")
        ax.set_yticks(range(len(group_labels)))
        ax.set_yticklabels(group_labels, fontsize=8.2)
        ax.tick_params(axis="both", which="both", length=0)
        ax.grid(False)

        cbar = fig.colorbar(im, ax=ax, fraction=0.032, pad=0.01, aspect=30)
        cbar.set_label("Presence rate", fontsize=8.5, labelpad=6)
        cbar.set_ticks([0.0, 0.25, 0.5, 0.75, 1.0])
        cbar.ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        cbar.ax.tick_params(labelsize=8)

        # Taxonomy section lines and right-side labels.
        section_order = [
            ("disability", "Disability"),
            ("gender", "Gender / LGBTQ"),
            ("lgbtq", "Gender / LGBTQ"),
            ("origin", "Origin"),
            ("politics", "Politics"),
            ("race", "Race"),
            ("religion", "Religion"),
        ]
        section_key_rank = {k: i for i, (k, _) in enumerate(section_order)}

        def _section_key(label: str) -> str:
            return str(label).split(":", 1)[0].strip()

        section_rows = {}
        for i, label in enumerate(group_labels):
            k = _section_key(label)
            section_rows.setdefault(k, []).append(i)

        ordered_sections = sorted(
            [(k, rows) for k, rows in section_rows.items()],
            key=lambda x: section_key_rank.get(x[0], 999),
        )

        # Draw lines between contiguous section blocks.
        for idx_sec in range(len(ordered_sections) - 1):
            last_row = max(ordered_sections[idx_sec][1])
            ax.axhline(last_row + 0.5, color="white", linewidth=1.5, zorder=4)

        ax2 = ax.twinx()
        ax2.set_ylim(ax.get_ylim())
        ax2.set_yticks([])
        section_display = {k: disp for k, disp in section_order}
        for k, rows in ordered_sections:
            mid = (min(rows) + max(rows)) / 2.0
            y = 1 - (mid / max(1, len(group_labels) - 1))
            ax2.text(
                1.02,
                y,
                section_display.get(k, k.title()),
                transform=ax2.transAxes,
                va="center",
                ha="left",
                fontsize=7.5,
                color="#555555",
            )

        legend_handles = [
            mpatches.Patch(
                facecolor="#E8E8E8",
                edgecolor="#cfcfcf",
                label="No glossary entries at this level",
            ),
            mpatches.Patch(
                facecolor="#CBD4D0",
                edgecolor="none",
                label="0% presence (entries exist, none found)",
            ),
            mpatches.FancyBboxPatch(
                (0, 0),
                1,
                1,
                boxstyle="square,pad=0",
                linewidth=1.5,
                edgecolor="#C0392B",
                facecolor="#d8d8d8",
                label="Type coverage < 100% (border + *)",
            ),
        ]
        ax.legend(
            handles=legend_handles,
            loc="lower left",
            bbox_to_anchor=(0, -0.13),
            fontsize=7.5,
            framealpha=0.9,
            ncol=1,
            handlelength=1.3,
            borderpad=0.6,
        )

        note = (
            "Cells marked * have partial type coverage (0 < type_coverage < 1.0). "
            "Gray cells are no-glossary combinations."
        )
        fig.text(
            0.012, 0.01, note, ha="left", va="bottom", fontsize=7.8, color="#555555"
        )

        _save_s5(fig, group_dir / "s5_grp_coverage_scatter.png")

    if not ann.empty:
        ann_plot = ann[
            (ann["report_level"] != "disability")
            | (ann["report_target"] == "unspecific")
        ].copy()
        ann_plot["label"] = (
            ann_plot["report_level"].astype(str)
            + ": "
            + ann_plot["report_target"].astype(str)
            + " ("
            + ann_plot["coding_level"].astype(str)
            + ")"
        )
        ann_plot = ann_plot.sort_values("label")

        # -- Annotation rates bar chart --
        fig, ax = plt.subplots(figsize=(6.97, 6))  # was (15, 6); matches gff textwidth_in
        fig.suptitle(
            "Annotation Quality — Correct vs. Failure Rate by Reporting Group",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        x = range(len(ann_plot))
        bar_w = 0.38
        ax.bar(
            x,
            ann_plot["correct_labeling_rate"],
            width=bar_w,
            color=_GREEN,
            label="Correct",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.bar(
            [v + bar_w for v in x],
            ann_plot["failure_rate"],
            width=bar_w,
            color=_ORANGE,
            label="Failure",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.set_xticks([v + bar_w / 2 for v in x])
        ax.set_xticklabels(ann_plot["label"], rotation=45, ha="right", fontsize=7.5)
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_ylabel("Rate", fontsize=9)
        _style_ax_s5(ax, grid_axis="y")
        ax.legend()
        _save_s5(fig, group_dir / "s5_grp_annotation_rates.png")

        # -- Case A/B stacked horizontal bar --
        d = _top_n_s5(ann_plot, "case_b_present_nonhateful", 20).sort_values(
            "case_b_present_nonhateful", ascending=True
        )
        fig, ax = plt.subplots(figsize=(6.97, max(6, 1 + 0.35 * len(d))))  # was width 13; matches gff textwidth_in
        fig.suptitle(
            "Annotation Quality — Case A vs. B Match Counts by Reporting Group",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        ax.barh(
            d["label"],
            d["case_a_present_hateful"],
            color=_GREEN,
            label="Case A (hateful)",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.barh(
            d["label"],
            d["case_b_present_nonhateful"],
            left=d["case_a_present_hateful"],
            color=_ORANGE,
            label="Case B (non-hateful)",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.set_xlabel("Match count", fontsize=9)
        _style_ax_s5(ax, grid_axis="x")
        ax.legend()
        _save_s5(fig, group_dir / "s5_grp_case_ab_counts.png")

    if not pair.empty:
        # -- DI ratio histograms --
        fig, axes = plt.subplots(1, 2, figsize=(6.97, 4.5))  # was (13, 5); matches gff textwidth_in
        fig.suptitle(
            "Disparity — Pairwise Coverage DI Ratio Distributions by Reporting Group",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for ax, col, label, color in [
            (axes[0], "presence_rate_di_ratio", "Presence DI", _BLUE),
            (axes[1], "type_coverage_di_ratio", "Type-Coverage DI", _TEAL),
        ]:
            ax.hist(
                pair[col].dropna(),
                bins=20,
                alpha=0.85,
                color=color,
                edgecolor="white",
                linewidth=0.5,
            )
            ax.axvline(
                DI_THRESHOLD,
                color=_RED,
                linestyle="--",
                linewidth=1.2,
                label="4/5 threshold",
            )
            ax.set_xlabel("DI ratio", fontsize=8)
            ax.set_ylabel("Count", fontsize=8)
            ax.set_title(label, fontweight="bold")
            _style_ax_s5(ax, grid_axis="y")
            ax.legend(fontsize=7)
        _save_s5(fig, group_dir / "s5_grp_di_histograms.png")

        # -- Worst DI + labeling gap panel --
        pair_plot = pair.copy()
        pair_plot["pair"] = (
            pair_plot["target_a"].astype(str)
            + " vs "
            + pair_plot["target_b"].astype(str)
        )
        # FIXED (see stage5_figures.py c92ea92-series fix): _top_n_s5() now takes
        # an explicit ascending= arg; "worst_di_ratio" (lower = more disparate =
        # worse) passes ascending=True so the n MOST disparate pairs are selected.
        left = _top_n_s5(pair_plot, "worst_di_ratio", 20, ascending=True).sort_values(
            "worst_di_ratio", ascending=True
        )
        right = _top_n_s5(pair_plot, "labeling_rate_gap_abs", 20).sort_values(
            "labeling_rate_gap_abs", ascending=True
        )

        fig, axes = plt.subplots(1, 2, figsize=(6.97, 7.5))  # was (17, 10); matches gff textwidth_in
        fig.suptitle(
            "Disparity — Worst Pairwise DI and Annotation Gap by Reporting Group",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        left_colors = [
            _RED if v < DI_THRESHOLD else _BLUE for v in left["worst_di_ratio"]
        ]
        axes[0].barh(
            left["pair"],
            left["worst_di_ratio"],
            color=left_colors,
            edgecolor="white",
            linewidth=0.5,
        )
        axes[0].axvline(
            DI_THRESHOLD,
            color=_RED,
            linestyle="--",
            linewidth=1.2,
            label=f"4/5 rule ({DI_THRESHOLD})",
        )
        axes[0].set_xlabel("Worst DI ratio", fontsize=8)
        axes[0].set_title("Worst DI pairs", fontweight="bold")
        _style_ax_s5(axes[0], grid_axis="x")
        axes[0].legend(fontsize=7)

        axes[1].barh(
            right["pair"],
            right["labeling_rate_gap_abs"],
            color=_RED,
            edgecolor="white",
            linewidth=0.5,
        )
        axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        axes[1].set_xlabel("|Correct rate gap|", fontsize=8)
        axes[1].set_title("Largest labeling gaps", fontweight="bold")
        _style_ax_s5(axes[1], grid_axis="x")
        _save_s5(fig, group_dir / "s5_grp_worst_di_and_label_gap.png")


In [ ]:
group_collapsed_figures(variant.out_s5 / "group_collapsed", variant.out_s4)


### ElSherief delta figures (`stage5_figures.elsherief_figures`)

**Function:** `elsherief_figures(els_dir, s4_dir)`

**Source TSVs read** (all **Stage 4c**, the ElSherief-subset delta rollup):
- `stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`
- `stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`
- `stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv`

**Output PNGs** (under `variant.out_s5/elsherief/`):
`s5_els_coverage_delta.png`, `s5_els_annotation_delta.png`,
`s5_els_pairwise_delta.png`.


In [ ]:
def elsherief_figures(els_dir: Path, s4_dir: Path) -> None:
    """Render delta figures comparing the union benchmark to ElSherief.

    Parameters
    ----------
    els_dir : Path
        Output directory for ElSherief delta figures.
    s4_dir : Path
        Input directory containing Stage 4 ElSherief artifacts.
    """
    cov = _read_s5(s4_dir / "elsherief/s4c_coverage_delta_union_vs_elsherief.tsv")
    ann = _read_s5(s4_dir / "elsherief/s4c_annotation_delta_union_vs_elsherief.tsv")
    pair = _read_s5(s4_dir / "elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv")

    def _diverging_bar(ax, labels, values, xlabel):
        """Draw a horizontal bar chart where sign is encoded by color."""
        colors = [_GREEN if v >= 0 else _RED for v in values.fillna(0)]
        ax.barh(labels, values, color=colors, edgecolor="white", linewidth=0.5)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_xlabel(xlabel, fontsize=9)
        _style_ax_s5(ax, grid_axis="x")

    if not cov.empty:
        d = cov.copy()
        d["label"] = (
            d["report_level"].astype(str) + ": " + d["report_target"].astype(str)
        )
        d = d.sort_values("presence_rate_delta_union_minus_elsherief", ascending=True)
        fig, ax = plt.subplots(figsize=(6.97, max(5, 1 + 0.3 * len(d))))
        fig.suptitle(
            "ElSherief Subset — Presence Rate \u0394 (Union \u2212 ElSherief)",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        _diverging_bar(
            ax,
            d["label"],
            d["presence_rate_delta_union_minus_elsherief"],
            "Presence rate \u0394",
        )
        _save_s5(fig, els_dir / "s5_els_coverage_delta.png")

    if not ann.empty:
        d = ann.copy()
        d["label"] = (
            d["report_level"].astype(str) + ": " + d["report_target"].astype(str)
        )
        d = d.sort_values(
            "correct_labeling_rate_delta_union_minus_elsherief", ascending=True
        )
        fig, ax = plt.subplots(figsize=(6.97, max(5, 1 + 0.3 * len(d))))
        fig.suptitle(
            "ElSherief Subset — Labeling Rate \u0394 (Union \u2212 ElSherief)",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        _diverging_bar(
            ax,
            d["label"],
            d["correct_labeling_rate_delta_union_minus_elsherief"],
            "Correct labeling rate \u0394",
        )
        _save_s5(fig, els_dir / "s5_els_annotation_delta.png")

    if not pair.empty:
        dx = pair["worst_di_ratio_delta_union_minus_elsherief"]
        dy = pair["label_gap_abs_delta_union_minus_elsherief"]
        fig, ax = plt.subplots(figsize=(5.0, 5.0))  # was (9, 7); scaled toward columnwidth-ish scatter
        fig.suptitle(
            "ElSherief Subset — Pairwise DI \u0394 vs. Label Gap \u0394",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        ax.scatter(
            dx, dy, alpha=0.75, s=50, c=_BLUE, edgecolors="#2c5f8a", linewidths=0.5
        )
        ax.axvline(0, color="#888888", linewidth=0.9, linestyle="--")
        ax.axhline(0, color="#888888", linewidth=0.9, linestyle="--")
        ax.set_xlabel("Worst DI \u0394 (union \u2212 ElSherief)", fontsize=9)
        ax.set_ylabel("|Label gap| \u0394 (union \u2212 ElSherief)", fontsize=9)
        _style_ax_s5(ax, grid_axis="both")
        _save_s5(fig, els_dir / "s5_els_pairwise_delta.png")


In [ ]:
elsherief_figures(variant.out_s5 / "elsherief", variant.out_s4)


## DEPRECATED exploratory pipeline -- `auditing/03_audit_visualizations.ipynb`

**This is a separate pipeline from `audit_pipeline/`**, with its own
Stage 00-03 notebooks (`auditing/00_coverage_audit.ipynb` through
`auditing/03_audit_visualizations.ipynb`, see `auditing/README.md`). Its
outputs currently live under `outputs/deprecated/{coverage,annotation,disparity}_audits/`
(originally `outputs/{coverage,annotation,disparity}_audits/` per the
notebook's own source -- **that top-level path no longer exists**, the data
was relocated under `deprecated/` at some point after this notebook was
last run). The cells below are otherwise byte-for-byte copies of the
original notebook's cells; only the three input directory assignments were
updated to point at the relocated data so this section can execute at all.
The output directory (`viz_output_dir`) is left exactly as the original
source wrote it (`outputs/audit_visualizations/`, not moved under
`deprecated/`) -- the source code never said to move it, so this notebook
doesn't move it either; running this section for the first time in a while
will (re)create that directory.

**Structural finding surfaced by cross-checking this pipeline against
`audit_pipeline` (see Step 3 below):** `outputs/deprecated/coverage_audits/audit_metrics.tsv`,
`outputs/deprecated/annotation_audits/annotation_quality.tsv`, and
`outputs/deprecated/disparity_audits/coverage_disparity.tsv` have **no
`coding_level` column at all** -- every metric here is pooled across all
Mendelsohn dogwhistle types (L1-L4) for a given `(taxonomy_level, target)`,
whereas every `audit_pipeline` stage stratifies by `coding_level`. A
same-named metric (`presence_rate`, `correct_labeling_rate`,
`presence_rate_di_ratio`, ...) computed by this deprecated pipeline is
therefore **not a like-for-like numeric re-derivation** of the corresponding
`audit_pipeline` metric -- it's a coarser aggregate. Step 3 logs this as
"not directly comparable -- different aggregation granularity" rather than
joining them and reporting a spurious mismatch.

**Figures produced** (all under `outputs/audit_visualizations/`):
`coverage_top20_token_frequency.png`, `coverage_presence_vs_type.png`,
`annotation_quality_top20_rates.png`, `annotation_case_ab_top15.png`,
`disparity_di_ratio_histograms.png`, `disparity_worst_di_top20.png`,
`annotation_disparity_label_gap_top20.png`, `cross_level_deltas.png`. A final
diagnostic (non-figure) cell prints high-confidence misclassified dogwhistle
forms; reproduced too since it's part of this file's real content.


### Setup + self-referential filtering (adapted paths, otherwise verbatim)


In [ ]:
DEPRECATED_DIR = LEGACY_OUTPUTS_DIR  # was: OUTPUTS_DIR / "deprecated" -- moved out of outputs/ entirely (2026-09-10)
coverage_dir = DEPRECATED_DIR / "coverage_audits"       # was: OUTPUTS_DIR / 'coverage_audits'
annotation_dir = DEPRECATED_DIR / "annotation_audits"    # was: OUTPUTS_DIR / 'annotation_audits'
disparity_dir = DEPRECATED_DIR / "disparity_audits"      # was: OUTPUTS_DIR / 'disparity_audits'
viz_output_dir = LEGACY_OUTPUTS_DIR / 'audit_visualizations'  # was: OUTPUTS_DIR / 'audit_visualizations' (2026-09-10)
viz_output_dir.mkdir(parents=True, exist_ok=True)

def read_tsv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    return pd.read_csv(path, sep='\t')

audit_metrics = read_tsv(coverage_dir / 'audit_metrics.tsv')
audit_detailed = read_tsv(coverage_dir / 'audit_detailed.tsv')
audit_matches = read_tsv(coverage_dir / 'audit_matches.tsv')
annotation_quality = read_tsv(annotation_dir / 'annotation_quality.tsv')
case_breakdown = read_tsv(annotation_dir / 'case_breakdown.tsv')
form_labeling = read_tsv(annotation_dir / 'form_labeling_detail.tsv')
coverage_disparity = read_tsv(disparity_dir / 'coverage_disparity.tsv')
annotation_disparity = read_tsv(disparity_dir / 'annotation_disparity.tsv')
cross_level_consistency = read_tsv(disparity_dir / 'cross_level_consistency.tsv')

# Define self-referential categories from form-level type annotations.
self_ref_mask = form_labeling['type'].fillna('').astype(str).str.contains('self-referential', case=False)
self_ref_rows = form_labeling[self_ref_mask].copy()
self_ref_pairs = set(
    zip(
        self_ref_rows['taxonomy_level'].astype(str),
        self_ref_rows['target'].astype(str),
    )
)
self_ref_triples = set(
    zip(
        self_ref_rows['taxonomy_level'].astype(str),
        self_ref_rows['target'].astype(str),
        self_ref_rows['dogwhistle'].astype(str),
    )
)

def exclude_self_ref_pairs(df: pd.DataFrame, level_col: str, target_col: str) -> pd.DataFrame:
    key = list(zip(df[level_col].astype(str), df[target_col].astype(str)))
    mask = [k not in self_ref_pairs for k in key]
    return df.loc[mask].copy()

def exclude_self_ref_triples(df: pd.DataFrame, level_col: str, target_col: str, dogwhistle_col: str) -> pd.DataFrame:
    key = list(zip(df[level_col].astype(str), df[target_col].astype(str), df[dogwhistle_col].astype(str)))
    mask = [k not in self_ref_triples for k in key]
    return df.loc[mask].copy()

# Build filtered views used by all visualizations.
audit_metrics_viz = exclude_self_ref_pairs(audit_metrics, 'taxonomy_level', 'target')
audit_detailed_viz = exclude_self_ref_triples(audit_detailed, 'taxonomy_level', 'target', 'dogwhistle')
audit_matches_viz = exclude_self_ref_triples(audit_matches, 'taxonomy_level', 'target', 'dogwhistle')
annotation_quality_viz = exclude_self_ref_pairs(annotation_quality, 'taxonomy_level', 'target')
case_breakdown_viz = exclude_self_ref_pairs(case_breakdown, 'taxonomy_level', 'target')
form_labeling_viz = exclude_self_ref_triples(form_labeling, 'taxonomy_level', 'target', 'dogwhistle')

coverage_disparity_viz = coverage_disparity[
    ~coverage_disparity.apply(
        lambda row: (
            (str(row['taxonomy_level']), str(row['target_a'])) in self_ref_pairs
            or (str(row['taxonomy_level']), str(row['target_b'])) in self_ref_pairs
        ),
        axis=1,
    )
].copy()

annotation_disparity_viz = annotation_disparity[
    ~annotation_disparity.apply(
        lambda row: (
            (str(row['taxonomy_level']), str(row['target_a'])) in self_ref_pairs
            or (str(row['taxonomy_level']), str(row['target_b'])) in self_ref_pairs
        ),
        axis=1,
    )
].copy()

cross_level_consistency_viz = cross_level_consistency[
    ~cross_level_consistency.apply(
        lambda row: (
            (str(row['level_from']), str(row['target'])) in self_ref_pairs
            or (str(row['level_to']), str(row['target'])) in self_ref_pairs
        ),
        axis=1,
    )
].copy()

coarse_levels = sorted(audit_metrics_viz['taxonomy_level'].dropna().astype(str).unique().tolist())

def make_facet_axes(levels: list[str], ncols: int = 3, panel_w: float = 5.0, panel_h: float = 4.0):
    n = len(levels)
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_w * ncols, panel_h * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, ncols).flatten()
    for idx in range(len(axes)):
        if idx >= n:
            axes[idx].axis('off')
    return fig, axes

print('Loaded datasets:')
for name, df in [
    ('audit_metrics', audit_metrics),
    ('audit_detailed', audit_detailed),
    ('audit_matches', audit_matches),
    ('annotation_quality', annotation_quality),
    ('case_breakdown', case_breakdown),
    ('form_labeling', form_labeling),
    ('coverage_disparity', coverage_disparity),
    ('annotation_disparity', annotation_disparity),
    ('cross_level_consistency', cross_level_consistency),
]:
    print(f'  - {name}: {len(df)} rows')

print('\nRemoved self-referential rows for visualizations:')
for name, full_df, filtered_df in [
    ('audit_metrics', audit_metrics, audit_metrics_viz),
    ('audit_detailed', audit_detailed, audit_detailed_viz),
    ('audit_matches', audit_matches, audit_matches_viz),
    ('annotation_quality', annotation_quality, annotation_quality_viz),
    ('case_breakdown', case_breakdown, case_breakdown_viz),
    ('form_labeling', form_labeling, form_labeling_viz),
    ('coverage_disparity', coverage_disparity, coverage_disparity_viz),
    ('annotation_disparity', annotation_disparity, annotation_disparity_viz),
    ('cross_level_consistency', cross_level_consistency, cross_level_consistency_viz),
]:
    print(f'  - {name}: removed {len(full_df) - len(filtered_df)} rows')

print('Facet levels:', coarse_levels)


### `coverage_top20_token_frequency.png`

**Output file:** `outputs/audit_visualizations/coverage_top20_token_frequency.png`

**Source TSV:** `audit_metrics.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 00 (deprecated `auditing/` numbering) of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.6, panel_h=4.6)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    plot_df = (
        audit_metrics_viz[audit_metrics_viz['taxonomy_level'] == level]
        .sort_values('token_frequency', ascending=False)
        .head(10)
        .copy()
    )

    if plot_df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: token frequency')
        ax.set_axis_off()
        continue

    plot_df = plot_df.sort_values('token_frequency', ascending=True)
    ax.barh(plot_df['target'], plot_df['token_frequency'], color='#2f6db5')
    ax.set_title(f'{level.title()}: token frequency')
    ax.set_xlabel('Token frequency')
    ax.set_ylabel('Target')

fig.suptitle(
    'Top token-frequency targets per coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'coverage_top20_token_frequency.png', dpi=180)
plt.show()


### `coverage_presence_vs_type.png`

**Output file:** `outputs/audit_visualizations/coverage_presence_vs_type.png`

**Source TSV:** `audit_metrics.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 00 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
norm = mpl.colors.Normalize(
    vmin=max(float(audit_metrics_viz['token_frequency'].min()), 1.0),
    vmax=max(float(audit_metrics_viz['token_frequency'].max()), 1.0),
)

fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.6, panel_h=4.6)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    level_df = audit_metrics_viz[audit_metrics_viz['taxonomy_level'] == level].copy()

    if level_df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: presence vs type coverage')
        ax.set_axis_off()
        continue

    sizes = np.clip(level_df['token_frequency'].to_numpy(), 1, None)
    sizes = 30 + 150 * (sizes / sizes.max())

    ax.scatter(
        level_df['presence_rate'],
        level_df['type_coverage'],
        s=sizes,
        c=level_df['token_frequency'],
        cmap='viridis',
        norm=norm,
        alpha=0.8,
        edgecolor='black',
        linewidth=0.3,
    )
    ax.set_title(f'{level.title()}: presence vs type coverage')
    ax.set_xlabel('Presence rate')
    ax.set_ylabel('Type coverage')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

fig.suptitle(
    'Coverage quality by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0.10, 1, 0.95])

sm = mpl.cm.ScalarMappable(norm=norm, cmap='viridis')
sm.set_array([])
left = axes[0].get_position().x0
right = axes[1].get_position().x1
cax = fig.add_axes([left, 0.055, right - left, 0.028])
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
cbar.set_label('Token frequency')

fig.savefig(viz_output_dir / 'coverage_presence_vs_type.png', dpi=180)
plt.show()


### `annotation_quality_top20_rates.png`

**Output file:** `outputs/audit_visualizations/annotation_quality_top20_rates.png`

**Source TSV:** `annotation_quality.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 01 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.2, panel_h=4.8)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    annot_plot = (
        annotation_quality_viz[annotation_quality_viz['taxonomy_level'] == level]
        .sort_values('total_matches', ascending=False)
        .head(8)
        .copy()
    )

    if annot_plot.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: annotation quality')
        ax.set_axis_off()
        continue

    x = np.arange(len(annot_plot))
    width = 0.4
    ax.bar(
        x - width / 2,
        annot_plot['correct_labeling_rate'],
        width=width,
        label='correct',
        color='#009E73',
    )
    ax.bar(
        x + width / 2,
        annot_plot['annotator_failure_ratio'],
        width=width,
        label='failure',
        color='#D55E00',
    )
    ax.set_xticks(x)
    ax.set_xticklabels(annot_plot['target'], rotation=50, ha='right')
    ax.set_ylim(0, 1)
    ax.set_title(f'{level.title()}: annotation quality')
    ax.set_ylabel('Rate')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    'Annotation quality rates by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'annotation_quality_top20_rates.png', dpi=180)
plt.show()


### `annotation_case_ab_top15.png`

**Output file:** `outputs/audit_visualizations/annotation_case_ab_top15.png`

**Source TSV:** `annotation_quality.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 01 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.8, panel_h=4.8)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    case_plot = (
        annotation_quality_viz[annotation_quality_viz['taxonomy_level'] == level]
        .sort_values('total_matches', ascending=False)
        .head(8)
        .sort_values('total_matches', ascending=True)
        .copy()
    )

    if case_plot.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: Case A/B composition')
        ax.set_axis_off()
        continue

    ax.barh(case_plot['target'], case_plot['case_a_present_hateful'], color='#009E73', label='Case A')
    ax.barh(
        case_plot['target'],
        case_plot['case_b_present_nonhateful'],
        left=case_plot['case_a_present_hateful'],
        color='#D55E00',
        label='Case B',
    )
    ax.set_title(f'{level.title()}: Case A/B composition')
    ax.set_xlabel('Matched count')
    ax.set_ylabel('Target')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    'Case A/B composition by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'annotation_case_ab_top15.png', dpi=180)
plt.show()


### `disparity_di_ratio_histograms.png`

**Output file:** `outputs/audit_visualizations/disparity_di_ratio_histograms.png`

**Source TSV:** `coverage_disparity.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 02 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.8, panel_h=4.4)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    level_df = coverage_disparity_viz[coverage_disparity_viz['taxonomy_level'] == level]

    if level_df.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: DI distribution')
        ax.set_axis_off()
        continue

    ax.hist(level_df['presence_rate_di_ratio'], bins=12, color='#4c78a8', alpha=0.65, label='presence DI')
    ax.hist(level_df['type_coverage_di_ratio'], bins=12, color='#72b7b2', alpha=0.65, label='type DI')
    ax.axvline(0.8, color='red', linestyle='--', linewidth=1.3)
    ax.set_title(f'{level.title()}: DI distribution')
    ax.set_xlabel('DI ratio')
    ax.set_ylabel('Pair count')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    'DI-ratio distributions by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'disparity_di_ratio_histograms.png', dpi=180)
plt.show()


### `disparity_worst_di_top20.png`

**Output file:** `outputs/audit_visualizations/disparity_worst_di_top20.png`

**Source TSV:** `coverage_disparity.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 02 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.4, panel_h=5.0)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    worst_di = coverage_disparity_viz[coverage_disparity_viz['taxonomy_level'] == level].copy()

    if worst_di.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: lowest DI comparisons')
        ax.set_axis_off()
        continue

    worst_di['pair'] = worst_di['target_a'] + ' vs ' + worst_di['target_b']
    worst_di['worst_di_ratio'] = worst_di[['presence_rate_di_ratio', 'type_coverage_di_ratio']].min(axis=1)
    worst_di = worst_di.sort_values('worst_di_ratio', ascending=True).head(8)

    ax.barh(worst_di['pair'], worst_di['worst_di_ratio'], color='#b279a2')
    ax.axvline(0.8, color='red', linestyle='--', linewidth=1.3)
    ax.set_title(f'{level.title()}: lowest DI comparisons')
    ax.set_xlabel('Worst DI ratio')
    ax.set_xlim(0, 1.05)

fig.suptitle(
    'Lowest DI-ratio comparisons by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'disparity_worst_di_top20.png', dpi=180)
plt.show()


### `annotation_disparity_label_gap_top20.png`

**Output file:** `outputs/audit_visualizations/annotation_disparity_label_gap_top20.png`

**Source TSV:** `annotation_disparity.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 02 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.4, panel_h=5.0)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    ann_gap = annotation_disparity_viz[annotation_disparity_viz['taxonomy_level'] == level].copy()

    if ann_gap.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: labeling-rate gaps')
        ax.set_axis_off()
        continue

    ann_gap['pair'] = ann_gap['target_a'] + ' vs ' + ann_gap['target_b']
    ann_gap = ann_gap.sort_values('labeling_rate_gap_abs', ascending=False).head(8).sort_values('labeling_rate_gap_abs')

    ax.barh(ann_gap['pair'], ann_gap['labeling_rate_gap_abs'], color='#e45756')
    ax.set_title(f'{level.title()}: labeling-rate gaps')
    ax.set_xlabel('Absolute correct-labeling-rate gap')

fig.suptitle(
    'Annotation quality gaps by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'annotation_disparity_label_gap_top20.png', dpi=180)
plt.show()


### `cross_level_deltas.png`

**Output file:** `outputs/audit_visualizations/cross_level_deltas.png`

**Source TSV:** `cross_level_consistency.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 02 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
if cross_level_consistency_viz.empty:
    print('cross_level_consistency is empty for this run after self-referential exclusion.')
else:
    fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.8, panel_h=4.8)

    for idx, level in enumerate(coarse_levels):
        ax = axes[idx]
        level_df = cross_level_consistency_viz[cross_level_consistency_viz['level_from'] == level].copy()

        if level_df.empty:
            ax.text(0.5, 0.5, 'No transitions from this level', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{level.title()}: cross-level deltas')
            ax.set_axis_off()
            continue

        labels = level_df['target'] + ' -> ' + level_df['level_to']
        y = np.arange(len(level_df))

        ax.barh(y - 0.2, level_df['presence_rate_delta'], height=0.38, color='#4c78a8', label='presence delta')
        ax.barh(y + 0.2, level_df['correct_labeling_rate_delta'], height=0.38, color='#54a24b', label='labeling delta')
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title(f'{level.title()}: cross-level deltas')
        ax.set_xlabel('Delta (to - from)')
        ax.legend(frameon=False, fontsize=8)

    fig.suptitle(
        'Cross-level deltas faceted by source coarse label (self-referential excluded)',
        y=0.99,
        fontsize=16,
        fontweight='semibold',
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(viz_output_dir / 'cross_level_deltas.png', dpi=180)
    plt.show()


### Diagnostic printout (not a figure) -- high-confidence misclassifications

**Reads:** `form_labeling_detail.tsv` (Stage 01) and `audit_matches.tsv`
(Stage 00) of the deprecated `auditing/` pipeline. Produces no image file;
reproduced here because it is part of the original notebook's real content
(guardrail: reproduce everything discovered in Step 1, not just the parts
that happen to be plots).


In [ ]:
print('=== High-confidence misclassifications (self-referential excluded) ===')

# Define high-confidence misclassification at form level.
# - Misclassification-prone: Case B exceeds Case A.
# - High-confidence: enough volume and low accuracy.
misclassified_forms = form_labeling_viz.copy()
misclassified_forms['total_matches'] = misclassified_forms['case_a_count'] + misclassified_forms['case_b_count']
misclassified_forms = misclassified_forms[
    (misclassified_forms['case_b_count'] > misclassified_forms['case_a_count'])
    & (misclassified_forms['total_matches'] >= 5)
    & (misclassified_forms['labeling_accuracy'] <= 0.20)
].copy()

misclassified_forms = misclassified_forms.sort_values(
    ['taxonomy_level', 'target', 'dogwhistle', 'labeling_accuracy', 'case_b_count'],
    ascending=[True, True, True, True, False],
)

if misclassified_forms.empty:
    print('No high-confidence misclassifications found under current filters.')
else:
    print(f"High-confidence misclassified forms: {len(misclassified_forms)}")
    print(
        misclassified_forms[
            [
                'taxonomy_level',
                'target',
                'dogwhistle',
                'type',
                'case_a_count',
                'case_b_count',
                'total_matches',
                'labeling_accuracy',
            ]
        ].to_string(index=False)
    )

    print('\n=== All matched non-hateful examples for these forms ===')
    all_examples = audit_matches_viz.merge(
        misclassified_forms[['taxonomy_level', 'target', 'dogwhistle']],
        on=['taxonomy_level', 'target', 'dogwhistle'],
        how='inner',
    )

    all_examples = all_examples[all_examples['binary_hate'] == 0].copy()
    all_examples = all_examples.drop_duplicates(
        ['taxonomy_level', 'target', 'dogwhistle', 'text_dedup_key']
    )
    all_examples = all_examples.sort_values(
        ['taxonomy_level', 'target', 'dogwhistle', 'text_dedup_key']
    )

    print(f"Total non-hateful matched examples: {len(all_examples)}")
    for _, row in all_examples.iterrows():
        print(f"- [{row['taxonomy_level']} | {row['target']} | {row['dogwhistle']}] {str(row['text'])}")


In [ ]:
print(f'Visualization images written to: {viz_output_dir}')


# Step 3 -- Cross-stage numerical consistency checks

Every quantity below is re-derived independently at **two or more**
pipeline stages/files. Each check loads the raw TSVs from every stage where
the quantity appears, joins on the shared keys, and asserts agreement within
`atol=1e-6` (`np.isclose(..., equal_nan=True)`, so two NaNs -- e.g. both
sides correctly suppressing an unstable small-n comparison -- count as
agreement, not a mismatch). Every mismatch is collected into one combined
table at the end rather than raising, per the task spec. The only numeric
literal used anywhere below is the `1e-6` comparison tolerance; every
"expected" value is read from a TSV or derived via `audit_pipeline.helpers`
functions imported from the real pipeline code (`map_reporting_group`,
`norm_target`) -- nothing is retyped from memory.

**What's being compared and why:**
- Stage 1 (`s1_coverage_by_level_target.tsv`, per raw target) vs Stage 4b
  (`s4b_coverage_by_level_group.tsv`, per reporting group) -- same
  `presence_rate` / `type_coverage`, computed by two independent code paths
  (Stage 1 aggregates over `s1_matches.tsv` directly in `stage1_coverage.py`;
  Stage 4 re-aggregates from the same match file via `compute_rollup()` in
  `stage4_rollup.py`). Restricted to reporting groups that are a 1:1 relabel
  of a raw target (`report_target == norm_target(target)`) -- collapsed
  groups like `LGB` / `Trans/NB` are legitimately many-to-one and are logged
  separately, not compared row-for-row.
- Stage 2 vs Stage 4b: same logic for `correct_labeling_rate` /
  `failure_rate` / `total_matches`.
- Stage 4a (`by_group`) vs Stage 4b (`by_level_group`): these two files
  differ only in whether `taxonomy_level` is kept as a group-by key. Since
  `map_reporting_group()` makes `report_level` a deterministic function of
  `taxonomy_level`, `taxonomy_level` is redundant once `report_level` /
  `report_target` are fixed -- so Stage 4a and Stage 4b should be exactly
  the same numbers, just with/without an extra (redundant) column. This is
  the concrete version of the "which by_* file is authoritative" ambiguity
  flagged throughout Step 2.
- Stage 3 pairwise DI (`target_a`/`target_b` = raw targets) vs Stage 4b and
  Stage 4a pairwise DI (`target_a`/`target_b` = reporting groups): three
  independently-built pairwise tables that different figures elsewhere in
  this notebook each treat as *the* source for "worst pairwise DI." Pairs
  are canonicalized (`tuple(sorted([a, b]))`) before joining since
  `itertools.combinations` order is not guaranteed to agree between the two
  independent pairwise-builders; only symmetric metrics (DI ratios and
  `*_gap_abs`, not signed `*_gap`) are compared for this reason.
- `generate_fig2_annotation_di_pairwise.py`'s own Stage 4a/Stage 4b merge:
  that script already detects overlapping `(target_a, target_b, coding_level)`
  keys between the two files and silently prefers Stage 4b -- but it never
  checks whether the two actually *agree*. This notebook adds that check.
- The deprecated `auditing/` pipeline vs `audit_pipeline`: logged as
  **not directly comparable** (see structural finding above) rather than
  joined, since the deprecated pipeline pools across coding levels.


In [ ]:
mismatches = []      # list[dict] -- rows for the final combined mismatch table
not_comparable = []  # list[str]  -- quantities with only one source, or structurally incompatible sources
checked = {}         # quantity label -> number of (group/pair, coding_level) rows compared

def _record_mismatches(quantity, df, key_cols, a_col, b_col, stage_a, stage_b):
    """Compare two already-joined columns; append any disagreement to `mismatches`."""
    n = len(df)
    checked[quantity] = checked.get(quantity, 0) + n
    if n == 0:
        return 0
    a = pd.to_numeric(df[a_col], errors="coerce")
    b = pd.to_numeric(df[b_col], errors="coerce")
    close = np.isclose(a.astype(float), b.astype(float), atol=1e-6, equal_nan=True)
    bad = df.loc[~close]
    a_bad = a.loc[~close]
    b_bad = b.loc[~close]
    for idx in bad.index:
        key = " / ".join(str(bad.loc[idx, c]) for c in key_cols)
        va, vb = a_bad.loc[idx], b_bad.loc[idx]
        delta = abs(va - vb) if pd.notna(va) and pd.notna(vb) else np.nan
        mismatches.append({
            "quantity": quantity,
            "group_or_pair": key,
            "coding_level": bad.loc[idx, "coding_level"] if "coding_level" in bad.columns else "",
            "stage_a": stage_a, "value_a": va,
            "stage_b": stage_b, "value_b": vb,
            "delta": delta,
        })
    return int((~close).sum())


## 3a -- Stage 1 vs Stage 4b (coverage: `presence_rate`, `type_coverage`)


In [ ]:
s1_cov = pd.read_csv(variant.out_s1 / "s1_coverage_by_level_target.tsv", sep="\t")
s4b_cov = pd.read_csv(variant.out_s4 / "by_level_group" / "s4b_coverage_by_level_group.tsv", sep="\t")

_mapped = s1_cov.apply(lambda r: map_reporting_group(r["taxonomy_level"], r["target"]), axis=1)
s1_cov = s1_cov.assign(
    report_level=[m.report_level for m in _mapped],
    report_target=[m.report_target for m in _mapped],
    report_include=[m.include for m in _mapped],
)
s1_cov_incl = s1_cov[s1_cov["report_include"]].copy()
identity_mask = (
    (s1_cov_incl["report_target"] == s1_cov_incl["target"].map(norm_target))
    & (s1_cov_incl["report_level"] == s1_cov_incl["taxonomy_level"].map(norm_target))
)
s1_identity = s1_cov_incl[identity_mask]
n_collapsed_s1 = int((~identity_mask).sum())

joined_a = s1_identity.merge(
    s4b_cov,
    on=["taxonomy_level", "coding_level", "report_level", "report_target"],
    suffixes=("_s1", "_s4b"),
    how="inner",
)
print(f"Stage1 vs Stage4b coverage: {len(joined_a)} matched rows joined "
      f"({n_collapsed_s1} Stage-1 rows excluded: many-to-one reporting-group "
      f"collapse, e.g. LGB/Trans-NB -- legitimate aggregation difference, not a discrepancy).")

for metric in ["presence_rate", "type_coverage"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage1 vs Stage4b)", joined_a,
        ["report_level", "report_target"],
        f"{metric}_s1", f"{metric}_s4b", "Stage 1", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")


## 3b -- Stage 2 vs Stage 4b (annotation: `correct_labeling_rate`, `failure_rate`, `total_matches`)


In [ ]:
s2_ann = pd.read_csv(variant.out_s2 / "s2_annotation_by_level_target.tsv", sep="\t")
s4b_ann = pd.read_csv(variant.out_s4 / "by_level_group" / "s4b_annotation_by_level_group.tsv", sep="\t")

_mapped2 = s2_ann.apply(lambda r: map_reporting_group(r["taxonomy_level"], r["target"]), axis=1)
s2_ann = s2_ann.assign(
    report_level=[m.report_level for m in _mapped2],
    report_target=[m.report_target for m in _mapped2],
    report_include=[m.include for m in _mapped2],
)
s2_incl = s2_ann[s2_ann["report_include"]].copy()
identity_mask2 = (
    (s2_incl["report_target"] == s2_incl["target"].map(norm_target))
    & (s2_incl["report_level"] == s2_incl["taxonomy_level"].map(norm_target))
)
s2_identity = s2_incl[identity_mask2]
n_collapsed_s2 = int((~identity_mask2).sum())

joined_b = s2_identity.merge(
    s4b_ann,
    on=["taxonomy_level", "coding_level", "report_level", "report_target"],
    suffixes=("_s2", "_s4b"),
    how="inner",
)
print(f"Stage2 vs Stage4b annotation: {len(joined_b)} matched rows joined "
      f"({n_collapsed_s2} Stage-2 rows excluded: many-to-one collapse).")

for metric in ["correct_labeling_rate", "failure_rate", "total_matches"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage2 vs Stage4b)", joined_b,
        ["report_level", "report_target"],
        f"{metric}_s2", f"{metric}_s4b", "Stage 2", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")


## 3c -- Stage 4a (`by_group`) vs Stage 4b (`by_level_group`)

Tests the hypothesis stated above: since `taxonomy_level` is a deterministic
function of `report_level` under `map_reporting_group()`, dropping it from
the group-by keys (as `by_group`/4a does) should produce numerically
identical rows to keeping it (as `by_level_group`/4b does).


In [ ]:
s4a_cov = pd.read_csv(variant.out_s4 / "by_group" / "s4a_coverage_by_group.tsv", sep="\t")
joined_c1 = s4a_cov.merge(
    s4b_cov, on=["coding_level", "report_level", "report_target"],
    suffixes=("_s4a", "_s4b"), how="outer", indicator=True,
)
n_onesided = int((joined_c1["_merge"] != "both").sum())
print(f"Stage4a vs Stage4b coverage: {len(joined_c1)} rows after outer join "
      f"({n_onesided} present on only one side -- 0 expected if the redundant-key hypothesis holds).")
joined_c1_both = joined_c1[joined_c1["_merge"] == "both"]
for metric in ["presence_rate", "type_coverage"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage4a vs Stage4b)", joined_c1_both,
        ["report_level", "report_target"],
        f"{metric}_s4a", f"{metric}_s4b", "Stage 4a", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")

s4a_ann = pd.read_csv(variant.out_s4 / "by_group" / "s4a_annotation_by_group.tsv", sep="\t")
joined_c2 = s4a_ann.merge(
    s4b_ann, on=["coding_level", "report_level", "report_target"],
    suffixes=("_s4a", "_s4b"), how="outer", indicator=True,
)
n_onesided2 = int((joined_c2["_merge"] != "both").sum())
print(f"Stage4a vs Stage4b annotation: {len(joined_c2)} rows after outer join "
      f"({n_onesided2} present on only one side).")
joined_c2_both = joined_c2[joined_c2["_merge"] == "both"]
for metric in ["correct_labeling_rate", "failure_rate", "total_matches"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage4a vs Stage4b)", joined_c2_both,
        ["report_level", "report_target"],
        f"{metric}_s4a", f"{metric}_s4b", "Stage 4a", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")


## 3d/3e -- Stage 3 pairwise DI vs Stage 4b and Stage 4a pairwise DI

Three independently-built pairwise tables (`s3_coverage_disparity.tsv` /
`s3_annotation_disparity.tsv` at raw-target granularity; the two Stage-4
`*_pairwise_disparity_*` files at reporting-group granularity). Only
orientation-symmetric metrics are compared (DI ratios and `*_gap_abs`
columns) since `target_a`/`target_b` order is not guaranteed to match across
independently-run `itertools.combinations` builds.


In [ ]:
s3_cov_disp = pd.read_csv(variant.out_s3 / "s3_coverage_disparity.tsv", sep="\t")
s3_ann_disp = pd.read_csv(variant.out_s3 / "s3_annotation_disparity.tsv", sep="\t")
s4b_pair = pd.read_csv(variant.out_s4 / "by_level_group" / "s4b_pairwise_disparity_by_level_group.tsv", sep="\t")
s4a_pair = pd.read_csv(variant.out_s4 / "by_group" / "s4a_pairwise_disparity_by_group.tsv", sep="\t")

def _map_pair_to_report(df, a_col="target_a", b_col="target_b", level_col="taxonomy_level"):
    df = df.copy()
    ma = df.apply(lambda r: map_reporting_group(r[level_col], r[a_col]), axis=1)
    mb = df.apply(lambda r: map_reporting_group(r[level_col], r[b_col]), axis=1)
    df["report_level"] = [m.report_level for m in ma]
    df["report_target_a"] = [m.report_target for m in ma]
    df["report_target_b"] = [m.report_target for m in mb]
    df["report_include"] = [m.include and n.include for m, n in zip(ma, mb)]
    df["identity_a"] = df["report_target_a"] == df[a_col].map(norm_target)
    df["identity_b"] = df["report_target_b"] == df[b_col].map(norm_target)
    df["pair_key"] = [tuple(sorted([a, b])) for a, b in zip(df["report_target_a"], df["report_target_b"])]
    return df

s3_cov_mapped = _map_pair_to_report(s3_cov_disp)
s3_ann_mapped = _map_pair_to_report(s3_ann_disp)

s3_cov_id = s3_cov_mapped[s3_cov_mapped["report_include"] & s3_cov_mapped["identity_a"] & s3_cov_mapped["identity_b"]]
s3_ann_id = s3_ann_mapped[s3_ann_mapped["report_include"] & s3_ann_mapped["identity_a"] & s3_ann_mapped["identity_b"]]
print(f"Stage3 coverage-disparity: {len(s3_cov_id)}/{len(s3_cov_mapped)} pairs kept for exact-match "
      f"checks (rest touch a collapsed reporting group on at least one side).")
print(f"Stage3 annotation-disparity: {len(s3_ann_id)}/{len(s3_ann_mapped)} pairs kept.")

s4b_pair2 = s4b_pair.copy()
s4b_pair2["pair_key"] = [tuple(sorted([a, b])) for a, b in zip(s4b_pair2["target_a"], s4b_pair2["target_b"])]
s4a_pair2 = s4a_pair.copy()
s4a_pair2["pair_key"] = [tuple(sorted([a, b])) for a, b in zip(s4a_pair2["target_a"], s4a_pair2["target_b"])]

# --- Stage 3 vs Stage 4b ---
joined_d_cov = s3_cov_id.merge(
    s4b_pair2, on=["taxonomy_level", "coding_level", "pair_key"],
    suffixes=("_s3", "_s4b"), how="inner",
)
print(f"\nStage3 coverage-disparity vs Stage4b pairwise: {len(joined_d_cov)} matched pairs.")
for metric in ["presence_rate_di_ratio", "type_coverage_di_ratio", "worst_di_ratio"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage3 vs Stage4b)", joined_d_cov,
        ["taxonomy_level", "coding_level", "pair_key"],
        f"{metric}_s3", f"{metric}_s4b", "Stage 3", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")

joined_d_ann = s3_ann_id.merge(
    s4b_pair2, on=["taxonomy_level", "coding_level", "pair_key"],
    suffixes=("_s3", "_s4b"), how="inner",
)
print(f"Stage3 annotation-disparity vs Stage4b pairwise: {len(joined_d_ann)} matched pairs.")
for metric in ["annotation_di_ratio", "labeling_rate_gap_abs", "failure_rate_gap_abs"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage3 vs Stage4b)", joined_d_ann,
        ["taxonomy_level", "coding_level", "pair_key"],
        f"{metric}_s3", f"{metric}_s4b", "Stage 3", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")

# --- Stage 3 vs Stage 4a (no taxonomy_level column on the 4a side; join on report_level instead) ---
joined_e_cov = s3_cov_id.merge(
    s4a_pair2, on=["coding_level", "report_level", "pair_key"],
    suffixes=("_s3", "_s4a"), how="inner",
)
print(f"\nStage3 coverage-disparity vs Stage4a pairwise: {len(joined_e_cov)} matched pairs.")
for metric in ["presence_rate_di_ratio", "type_coverage_di_ratio", "worst_di_ratio"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage3 vs Stage4a)", joined_e_cov,
        ["coding_level", "report_level", "pair_key"],
        f"{metric}_s3", f"{metric}_s4a", "Stage 3", "Stage 4a",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")

joined_e_ann = s3_ann_id.merge(
    s4a_pair2, on=["coding_level", "report_level", "pair_key"],
    suffixes=("_s3", "_s4a"), how="inner",
)
print(f"Stage3 annotation-disparity vs Stage4a pairwise: {len(joined_e_ann)} matched pairs.")
for metric in ["annotation_di_ratio", "labeling_rate_gap_abs", "failure_rate_gap_abs"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage3 vs Stage4a)", joined_e_ann,
        ["coding_level", "report_level", "pair_key"],
        f"{metric}_s3", f"{metric}_s4a", "Stage 3", "Stage 4a",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")


## 3f -- `generate_fig2_annotation_di_pairwise.py`'s own Stage 4a/4b overlap

That script's `load_and_merge()` already finds `(target_a, target_b,
coding_level)` keys present as stable rows in *both* `s4b` and `s4a`, and
silently keeps the `s4b` copy while dropping `s4a`'s -- without ever
checking whether the two actually agree. Reusing its own `KEY_COLS` and
`_filter_stable` to check that now.


In [ ]:
# generate_fig2_annotation_di_pairwise.py is archived (deprecated/), and the
# fig2_annotation_di_by_pair_level section that used to define KEY_COLS/
# _filter_stable now lives in figures_consolidated.ipynb, not this notebook --
# defined locally here instead (identical to that section's copy).
FIG2_KEY_COLS = ["target_a", "target_b", "coding_level"]

def _filter_stable(df: pd.DataFrame) -> pd.DataFrame:
    return df[~df["unstable_small_n"].astype(bool)].dropna(subset=["annotation_di_ratio"])

fig2_filter_stable = _filter_stable

s4b_stable = fig2_filter_stable(s4b_pair.copy())
s4a_stable = fig2_filter_stable(s4a_pair.copy())
overlap_keys = set(map(tuple, s4b_stable[FIG2_KEY_COLS].values)) & set(map(tuple, s4a_stable[FIG2_KEY_COLS].values))
print(f"{len(overlap_keys)} (target_a, target_b, coding_level) key(s) are stable in both Stage 4a and Stage 4b.")

if overlap_keys:
    s4b_ov = s4b_stable[s4b_stable[FIG2_KEY_COLS].apply(tuple, axis=1).isin(overlap_keys)]
    s4a_ov = s4a_stable[s4a_stable[FIG2_KEY_COLS].apply(tuple, axis=1).isin(overlap_keys)]
    joined_f = s4a_ov.merge(s4b_ov, on=FIG2_KEY_COLS, suffixes=("_s4a", "_s4b"))
    n_bad = _record_mismatches(
        "annotation_di_ratio (Stage4a/Stage4b overlap keys used by generate_fig2_annotation_di_pairwise.py)",
        joined_f, list(FIG2_KEY_COLS),
        "annotation_di_ratio_s4a", "annotation_di_ratio_s4b", "Stage 4a", "Stage 4b",
    )
    print(f"  annotation_di_ratio: {n_bad} mismatch(es) among overlap keys")
else:
    print("  no overlapping stable keys found for this pipeline run.")


## 3g -- Deprecated `auditing/` pipeline vs `audit_pipeline`

No join is attempted here (see structural finding in the `auditing/`
section above): the deprecated pipeline's coverage/annotation/disparity
files have no `coding_level` column, so every metric is pooled across L1-L4
for a `(taxonomy_level, target)` pair -- not a like-for-like re-derivation
of any single `audit_pipeline` stage's per-coding-level metric. Logged as
not-comparable rather than fabricating a join that would either fan out or
silently misrepresent a pooled number as a per-level one.


In [ ]:
not_comparable.append(
    "presence_rate / type_coverage / correct_labeling_rate / failure_rate / "
    "presence_rate_di_ratio / annotation_di_ratio (deprecated auditing/ Stage 00-02 "
    "vs audit_pipeline Stage 1-4): the deprecated pipeline's coverage_disparity.tsv, "
    "annotation_quality.tsv, and audit_metrics.tsv have NO coding_level column -- "
    "every metric is pooled across all Mendelsohn dogwhistle types (L1-L4) for a "
    "(taxonomy_level, target) pair, while every audit_pipeline stage stratifies by "
    "coding_level. A row-level join would fan out (1 deprecated row vs up to 4 "
    "audit_pipeline rows) or require independently reproducing the deprecated "
    "pipeline's pooling logic, which is out of scope for this pass. Logged as "
    "single-source / not directly comparable rather than fabricating a join."
)
for note in not_comparable:
    print("- " + note)


## Step 3 result -- combined cross-stage mismatch table

Empty is the success condition.


In [ ]:
mismatch_df = pd.DataFrame(
    mismatches,
    columns=["quantity", "group_or_pair", "coding_level", "stage_a", "value_a", "stage_b", "value_b", "delta"],
)
if not mismatch_df.empty:
    mismatch_df = mismatch_df.sort_values("delta", ascending=False, na_position="last").reset_index(drop=True)

total_checked = sum(checked.values())
print("=" * 88)
print("CROSS-STAGE CONSISTENCY SUMMARY")
print("=" * 88)
print(f"Quantity x group/pair x coding_level combinations checked: {total_checked} "
      f"across {len(checked)} quantity comparisons.")
for q, n in checked.items():
    print(f"  - {q}: {n} rows compared")
print(f"\nQuantities logged as single-source / not directly comparable: {len(not_comparable)}")
print(f"\nMismatches found (|delta| > 1e-6): {len(mismatch_df)}")
if mismatch_df.empty:
    print("EMPTY -- every cross-stage-computed quantity checked above agrees within 1e-6.")
else:
    print("See table below, sorted by delta descending.")
    with pd.option_context("display.max_rows", None, "display.width", 160):
        print(mismatch_df.to_string(index=False))

mismatch_df


## Archival note

The summary below was written when this content was still part of a single
`figures_consolidated.ipynb` covering both the paper figures and these
legacy pipelines. Kept verbatim as a historical record of that original
consolidation exercise (it inventories `generate_figures_final.py` and
`generate_fig2_annotation_di_pairwise.py` too, which now live in the other
notebook) -- not re-written for the split, since nothing it describes has
changed, only which file it's written down in.

# Step 4 -- Final summary

## Step 1 inventory -- every figure-generation file found in the repo

| File | Figures produced | Input files read (exact paths) |
|---|---|---|
| `audit_pipeline/stage5_figures.py` | 16 PNGs across `level_stratified_figures` (8), `group_collapsed_figures` (5), `elsherief_figures` (3) | `stage1/s1_coverage_by_level_target.tsv`, `stage2/s2_annotation_by_level_target.tsv`, `stage3/s3_coverage_disparity.tsv`, `stage3/s3_annotation_disparity.tsv`, `stage3/s3_cross_level_consistency.tsv`, `stage4/by_level_group/s4b_coverage_by_level_group.tsv`, `stage4/by_group/s4a_annotation_by_group.tsv`, `stage4/by_group/s4a_pairwise_disparity_by_group.tsv`, `stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv` |
| `generate_figures_final.py` (repo root) | 13 PDFs: `fig1`-`fig7`, `appA`-`appG` | `stage1/s1_coverage_by_level_target.tsv`, `unioned_data/06_glossary_label_reference.tsv`, `stage2/s2_annotation_by_level_target.tsv`, `stage3/s3_cross_level_consistency.tsv`, `stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`, `stage4/by_level_group/s4b_coverage_by_level_group.tsv`, `stage4/by_level_group/s4b_annotation_by_level_group.tsv`, `stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv` |
| `generate_fig2_annotation_di_pairwise.py` (repo root) | 1 PDF: `fig02_annotation_di_by_pair_level.pdf` (or `_v2` if it already exists) | `stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`, `stage4/by_group/s4a_pairwise_disparity_by_group.tsv` |
| `auditing/03_audit_visualizations.ipynb` (deprecated pipeline) | 8 PNGs + 1 diagnostic printout | `outputs/deprecated/coverage_audits/{audit_metrics,audit_detailed,audit_matches}.tsv`, `outputs/deprecated/annotation_audits/{annotation_quality,form_labeling_detail}.tsv`, `outputs/deprecated/disparity_audits/{coverage_disparity,annotation_disparity,cross_level_consistency}.tsv` (originally pointed at `outputs/{coverage,annotation,disparity}_audits/`, which no longer exist at that path) |

**Total: 4 files, 38 figures + 1 diagnostic printout, all reproduced in this notebook.**

## Step 2 provenance -- figure to source file to pipeline stage

See the markdown cell immediately preceding each figure's code cell above
for the full per-figure provenance block (output filename, exact source
TSV path(s), pipeline stage). Section-level summary:

| Section | Pipeline stage(s) used |
|---|---|
| Level-stratified figures (`stage5_figures.py`) | Stage 1, Stage 2, Stage 3 |
| Group-collapsed figures (`stage5_figures.py`) | Stage 4b (coverage only), Stage 4a (annotation + pairwise) |
| ElSherief delta figures (`stage5_figures.py`) | Stage 4c |
| `generate_figures_final.py` (`fig1`-`fig7`, `appA`-`appG`) | Stage 1, Stage 2, Stage 3, Stage 4b, Stage 4c, and one upstream preprocessing file |
| `generate_fig2_annotation_di_pairwise.py` | Stage 4a + Stage 4b, merged |
| `auditing/03_audit_visualizations.ipynb` | Deprecated pipeline's own Stage 00-02 (not `audit_pipeline` numbering) |

**Key ambiguity surfaced (the reason this task exists):** a "worst pairwise
DI" figure exists reading **Stage 3** (`stage5_figures.level_stratified_figures`
-> `s5_lev_disparity_worst_di.png`), a second reading **Stage 4a**
(`stage5_figures.group_collapsed_figures` -> `s5_grp_worst_di_and_label_gap.png`),
and a third reading **Stage 4b** (`generate_figures_final.fig5_worst_di_by_pair_level`
/ `appD_worst_di_and_label_gap_pooled`, and `generate_fig2_annotation_di_pairwise.py`,
which merges 4a+4b itself). None of the four figure-generation files agree
on a single canonical pairwise-DI source, and the Stage 4b file
(`by_level_group`) -- the one a human manually spot-checking numbers would
most likely reach for first, since it's the only one whose name mentions
both "level" and "group" -- is used by exactly one of the four call sites
that produce a "worst DI" figure. Step 3 above numerically confirms whether
these different sources actually agree.

## Step 3 result -- cross-stage consistency

See the "Step 3 result" cell above for the live, executed numbers (row
counts checked, mismatch count, full mismatch table if non-empty). That
cell is authoritative; this paragraph is a static description of what was
checked: Stage 1 vs Stage 4b coverage, Stage 2 vs Stage 4b annotation,
Stage 4a vs Stage 4b (coverage and annotation), Stage 3 vs Stage 4b pairwise
DI, Stage 3 vs Stage 4a pairwise DI, and `generate_fig2_annotation_di_pairwise.py`'s
own Stage 4a/4b overlap keys -- seven independent cross-checks in total. The
deprecated `auditing/` pipeline was found to be structurally incompatible
with `audit_pipeline` (no `coding_level` stratification) and is logged as
not-comparable rather than joined.

## Known bugs fixed

Both occurrences of the `_top_n_s5` sort-direction bug described in the task
brief have been fixed. `_top_n_s5` now takes an explicit `ascending=`
argument (default `False`, unchanged for every other call site -- token
frequency, label gaps, etc., where higher = worse); both
`"worst_di_ratio"` call sites now pass `ascending=True` so the n MOST
disparate pairs are selected, not the least:

1. **Cell 6** (Level-stratified figures / `level_stratified_figures`),
   `d = _top_n_s5(sub, "worst_di_ratio", 8, ascending=True).sort_values(`
   -- affects `s5_lev_disparity_worst_di.png`.
2. **Cell 9** (Group-collapsed figures / `group_collapsed_figures`),
   `left = _top_n_s5(pair_plot, "worst_di_ratio", 20, ascending=True).sort_values(`
   -- affects `s5_grp_worst_di_and_label_gap.png`.

(Cell indices are 0-based over the full notebook's cell list, counting both
markdown and code cells, as saved in this build.)

## Other surfaced (not fixed) discrepancies

- `generate_fig2_annotation_di_pairwise.py`'s docstring says it matches
  "Figure 9 (fig10_worst_di_by_pair_level.pdf)" but its own output file is
  named `fig02_annotation_di_by_pair_level.pdf` -- a `fig2` name that also
  collides (in name only, not content) with `generate_figures_final.py`'s
  unrelated `fig01_annotation_rates_by_level.pdf`.
- `generate_figures_final.py` and `generate_fig2_annotation_di_pairwise.py`
  are not `PipelineVariant`-aware (no tier1+2 robustness-check support),
  unlike `stage5_figures.py`.
- `generate_fig2_annotation_di_pairwise.py` hardcodes a local
  `DI_THRESHOLD = 0.80` instead of importing `audit_pipeline.config.DI_THRESHOLD`
  (numerically identical, but a third independent definition of the same
  constant).
- The deprecated `auditing/` pipeline pools all metrics across coding levels
  (no `coding_level` column anywhere in its output), unlike every
  `audit_pipeline` stage.

## Success checklist

- [x] Notebook runs top-to-bottom with a fresh kernel, no errors (verified
      via a redirected-output copy -- see build notes; this delivered copy
      targets the real production paths and was not itself executed, so
      that no existing figure artifact is modified as a side effect of
      building this notebook).
- [x] Every figure produced by any discovered script is reproduced here
      under the same output filename.
- [x] Every figure cell is preceded by a provenance markdown block.
- [x] The cross-stage consistency section runs and produces a summary table
      for every quantity with a second independent source (7 checks; see
      Step 3 above for the live mismatch count).
- [x] Zero hardcoded "expected" values -- all Step 3 comparisons read from
      TSVs or `audit_pipeline` code; the only numeric literal introduced is
      the `1e-6` comparison tolerance.
- [x] Both `_top_n_s5` "worst DI" bug sites fixed above (see "Known bugs
      fixed" section) with cell/line pointers.
- [x] No existing file modified -- this notebook is the only new file this
      task produced.


In [ ]:
from IPython.display import Markdown, display

summary_text = '# Step 4 -- Final summary\n\n## Step 1 inventory -- every figure-generation file found in the repo\n\n| File | Figures produced | Input files read (exact paths) |\n|---|---|---|\n| `audit_pipeline/stage5_figures.py` | 16 PNGs across `level_stratified_figures` (8), `group_collapsed_figures` (5), `elsherief_figures` (3) | `stage1/s1_coverage_by_level_target.tsv`, `stage2/s2_annotation_by_level_target.tsv`, `stage3/s3_coverage_disparity.tsv`, `stage3/s3_annotation_disparity.tsv`, `stage3/s3_cross_level_consistency.tsv`, `stage4/by_level_group/s4b_coverage_by_level_group.tsv`, `stage4/by_group/s4a_annotation_by_group.tsv`, `stage4/by_group/s4a_pairwise_disparity_by_group.tsv`, `stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv` |\n| `generate_figures_final.py` (repo root) | 13 PDFs: `fig1`-`fig7`, `appA`-`appG` | `stage1/s1_coverage_by_level_target.tsv`, `unioned_data/06_glossary_label_reference.tsv`, `stage2/s2_annotation_by_level_target.tsv`, `stage3/s3_cross_level_consistency.tsv`, `stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`, `stage4/by_level_group/s4b_coverage_by_level_group.tsv`, `stage4/by_level_group/s4b_annotation_by_level_group.tsv`, `stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv` |\n| `generate_fig2_annotation_di_pairwise.py` (repo root) | 1 PDF: `fig02_annotation_di_by_pair_level.pdf` (or `_v2` if it already exists) | `stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`, `stage4/by_group/s4a_pairwise_disparity_by_group.tsv` |\n| `auditing/03_audit_visualizations.ipynb` (deprecated pipeline) | 8 PNGs + 1 diagnostic printout | `outputs/deprecated/coverage_audits/{audit_metrics,audit_detailed,audit_matches}.tsv`, `outputs/deprecated/annotation_audits/{annotation_quality,form_labeling_detail}.tsv`, `outputs/deprecated/disparity_audits/{coverage_disparity,annotation_disparity,cross_level_consistency}.tsv` (originally pointed at `outputs/{coverage,annotation,disparity}_audits/`, which no longer exist at that path) |\n\n**Total: 4 files, 38 figures + 1 diagnostic printout, all reproduced in this notebook.**\n\n## Step 2 provenance -- figure to source file to pipeline stage\n\nSee the markdown cell immediately preceding each figure\'s code cell above\nfor the full per-figure provenance block (output filename, exact source\nTSV path(s), pipeline stage). Section-level summary:\n\n| Section | Pipeline stage(s) used |\n|---|---|\n| Level-stratified figures (`stage5_figures.py`) | Stage 1, Stage 2, Stage 3 |\n| Group-collapsed figures (`stage5_figures.py`) | Stage 4b (coverage only), Stage 4a (annotation + pairwise) |\n| ElSherief delta figures (`stage5_figures.py`) | Stage 4c |\n| `generate_figures_final.py` (`fig1`-`fig7`, `appA`-`appG`) | Stage 1, Stage 2, Stage 3, Stage 4b, Stage 4c, and one upstream preprocessing file |\n| `generate_fig2_annotation_di_pairwise.py` | Stage 4a + Stage 4b, merged |\n| `auditing/03_audit_visualizations.ipynb` | Deprecated pipeline\'s own Stage 00-02 (not `audit_pipeline` numbering) |\n\n**Key ambiguity surfaced (the reason this task exists):** a "worst pairwise\nDI" figure exists reading **Stage 3** (`stage5_figures.level_stratified_figures`\n-> `s5_lev_disparity_worst_di.png`), a second reading **Stage 4a**\n(`stage5_figures.group_collapsed_figures` -> `s5_grp_worst_di_and_label_gap.png`),\nand a third reading **Stage 4b** (`generate_figures_final.fig5_worst_di_by_pair_level`\n/ `appD_worst_di_and_label_gap_pooled`, and `generate_fig2_annotation_di_pairwise.py`,\nwhich merges 4a+4b itself). None of the four figure-generation files agree\non a single canonical pairwise-DI source, and the Stage 4b file\n(`by_level_group`) -- the one a human manually spot-checking numbers would\nmost likely reach for first, since it\'s the only one whose name mentions\nboth "level" and "group" -- is used by exactly one of the four call sites\nthat produce a "worst DI" figure. Step 3 above numerically confirms whether\nthese different sources actually agree.\n\n## Step 3 result -- cross-stage consistency\n\nSee the "Step 3 result" cell above for the live, executed numbers (row\ncounts checked, mismatch count, full mismatch table if non-empty). That\ncell is authoritative; this paragraph is a static description of what was\nchecked: Stage 1 vs Stage 4b coverage, Stage 2 vs Stage 4b annotation,\nStage 4a vs Stage 4b (coverage and annotation), Stage 3 vs Stage 4b pairwise\nDI, Stage 3 vs Stage 4a pairwise DI, and `generate_fig2_annotation_di_pairwise.py`\'s\nown Stage 4a/4b overlap keys -- seven independent cross-checks in total. The\ndeprecated `auditing/` pipeline was found to be structurally incompatible\nwith `audit_pipeline` (no `coding_level` stratification) and is logged as\nnot-comparable rather than joined.\n\n## Known bugs fixed\n\nBoth occurrences of the `_top_n_s5` sort-direction bug described in the task\nbrief have been fixed. `_top_n_s5` now takes an explicit `ascending=`\nargument (default `False`, unchanged for every other call site -- token\nfrequency, label gaps, etc., where higher = worse); both\n`"worst_di_ratio"` call sites now pass `ascending=True` so the n MOST\ndisparate pairs are selected, not the least:\n\n1. **Cell 6** (Level-stratified figures / `level_stratified_figures`),\n   `d = _top_n_s5(sub, "worst_di_ratio", 8, ascending=True).sort_values(`\n   -- affects `s5_lev_disparity_worst_di.png`.\n2. **Cell 9** (Group-collapsed figures / `group_collapsed_figures`),\n   `left = _top_n_s5(pair_plot, "worst_di_ratio", 20, ascending=True).sort_values(`\n   -- affects `s5_grp_worst_di_and_label_gap.png`.\n\n(Cell indices are 0-based over the full notebook\'s cell list, counting both\nmarkdown and code cells, as saved in this build.)\n\n## Other surfaced (not fixed) discrepancies\n\n- `generate_fig2_annotation_di_pairwise.py`\'s docstring says it matches\n  "Figure 9 (fig10_worst_di_by_pair_level.pdf)" but its own output file is\n  named `fig02_annotation_di_by_pair_level.pdf` -- a `fig2` name that also\n  collides (in name only, not content) with `generate_figures_final.py`\'s\n  unrelated `fig01_annotation_rates_by_level.pdf`.\n- `generate_figures_final.py` and `generate_fig2_annotation_di_pairwise.py`\n  are not `PipelineVariant`-aware (no tier1+2 robustness-check support),\n  unlike `stage5_figures.py`.\n- `generate_fig2_annotation_di_pairwise.py` hardcodes a local\n  `DI_THRESHOLD = 0.80` instead of importing `audit_pipeline.config.DI_THRESHOLD`\n  (numerically identical, but a third independent definition of the same\n  constant).\n- The deprecated `auditing/` pipeline pools all metrics across coding levels\n  (no `coding_level` column anywhere in its output), unlike every\n  `audit_pipeline` stage.\n\n## Success checklist\n\n- [x] Notebook runs top-to-bottom with a fresh kernel, no errors (verified\n      via a redirected-output copy -- see build notes; this delivered copy\n      targets the real production paths and was not itself executed, so\n      that no existing figure artifact is modified as a side effect of\n      building this notebook).\n- [x] Every figure produced by any discovered script is reproduced here\n      under the same output filename.\n- [x] Every figure cell is preceded by a provenance markdown block.\n- [x] The cross-stage consistency section runs and produces a summary table\n      for every quantity with a second independent source (7 checks; see\n      Step 3 above for the live mismatch count).\n- [x] Zero hardcoded "expected" values -- all Step 3 comparisons read from\n      TSVs or `audit_pipeline` code; the only numeric literal introduced is\n      the `1e-6` comparison tolerance.\n- [x] Both `_top_n_s5` "worst DI" bug sites fixed above (see "Known bugs\n      fixed" section) with cell/line pointers.\n- [x] No existing file modified -- this notebook is the only new file this\n      task produced.\n'

print(summary_text)  # echoed to stdout per the task spec
display(Markdown(summary_text))
